# Matched Pairwise Contrastive Learning

This outcome-free experiment asks whether a regularized linear direction learned from matched positive-minus-negative embedding differences can rank a beneficiary above its otherwise similar adverse-exposure counterpart. It follows the unsuccessful 10D centroid test but represents a genuinely different learning rule. Post-event returns remain unloaded.

In [ ]:
from pathlib import Path
import importlib,sys,time
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
ROOT=Path.cwd()
if ROOT.name=='geopolitical_exposure':GEO,ROOT=ROOT,ROOT.parent
else:GEO=ROOT/'geopolitical_exposure'
sys.path.insert(0,str(ROOT))
RESULTS=GEO/'matched_contrastive';RESULTS.mkdir(parents=True,exist_ok=True)
import geopolitical_exposure.src.directional_centroid as dc
import geopolitical_exposure.src.matched_contrastive as mc
importlib.reload(dc);importlib.reload(mc)
SEEDS=GEO/'embedding_expansion_v2/directional_firm_channel_seeds.parquet'
print('Event returns loaded: False')

## 1. Reconstruct frozen firm labels and matching inputs

The same 29 positive-only and 55 negative-only firms are used. Matching sees only SIC2 and six pre-event characteristics; it never sees embeddings, model scores, or returns.

In [ ]:
labels,_=dc.build_firm_labels(pd.read_parquet(SEEDS))
representations=dc.load_representations(ROOT)
reference=representations['all_s12_rs_binary'][['permco','firm_name']]
characteristics_raw=pd.read_parquet(ROOT/'remaining_benchmarks/return_comovement/characteristics/2021q4.parquet')
industry=pd.read_parquet(GEO/'event_data/crsp_point_in_time_industry_2021q4.parquet')
pairs,balance=mc.build_matched_pairs(labels,characteristics_raw,industry)
pairs.to_parquet(RESULTS/'frozen_matched_pairs.parquet',index=False)
balance.to_parquet(RESULTS/'matching_balance.parquet',index=False)
display(pd.DataFrame([{'pairs':len(pairs),'unique_positive_firms':pairs.positive_firm_id.nunique(),'unique_negative_firms':pairs.negative_firm_id.nunique(),'exact_sic2_pairs':int(pairs.same_sic2.sum()),'same_sic_division_pairs':int(pairs.same_sic_division.sum()),'median_characteristic_distance':pairs.characteristic_distance.median()}]))
display(balance)

## 2. Freeze the contrastive design

The primary model is All-S12 RS-Binary 10D with ridge penalty 1. Each pair contributes a difference vector $x_{positive}-x_{negative}$. The model learns a direction that maps training differences toward +1. Both firms in the evaluated pair are held out.

In [ ]:
design=mc.write_design(RESULTS/'design.json',pairs)
display(pd.DataFrame([design]))

## 3. Difference-vector alignment

Before supervised fitting, we ask whether independently matched positive-minus-negative vectors point in a common direction. Strong positive cosine alignment would support a shared translation axis; near-zero alignment would imply heterogeneous pair directions.

In [ ]:
alignment=[]
for name,frame in representations.items():
    row=mc.difference_alignment(frame,pairs,10);row['representation']=name;alignment.append(row)
alignment=pd.concat(alignment,ignore_index=True);alignment.to_parquet(RESULTS/'difference_alignment_10d.parquet',index=False)
display(alignment.sort_values('mean_difference_cosine',ascending=False))

## 4. Leave-one-pair-out contrastive validation

For each of 29 pairs, the model is fit on the other 28 difference vectors. Success means the held-out positive firm ranks above its matched negative firm. Pair accuracy is the primary metric.

In [ ]:
metric_frames=[];prediction_frames=[]
for name,frame in representations.items():
    metrics,predictions=mc.evaluate_representation(frame,pairs,name,penalty=1.0)
    metric_frames.append(metrics);prediction_frames.append(predictions)
metrics=pd.concat(metric_frames,ignore_index=True);predictions=pd.concat(prediction_frames,ignore_index=True)
metrics.to_parquet(RESULTS/'loo_pair_metrics_all_dimensions.parquet',index=False)
predictions.to_parquet(RESULTS/'loo_pair_predictions_all_dimensions.parquet',index=False)
display(metrics.query('dimension==10').sort_values('pair_accuracy',ascending=False))

## 5. Primary sign-flip permutation test

The null independently reverses positive/negative orientation within each fixed pair. Every permutation reruns all leave-one-pair-out fits. This preserves pair composition and asks whether the observed orientation is systematically recoverable.

In [ ]:
started=time.perf_counter()
primary_summary,primary_null=mc.sign_flip_permutation(representations['all_s12_rs_binary'],pairs,'all_s12_rs_binary',dimension=10,penalty=1.0,repetitions=5000)
primary_summary['seconds']=time.perf_counter()-started
primary_summary.to_parquet(RESULTS/'primary_permutation_summary.parquet',index=False)
primary_null.to_parquet(RESULTS/'primary_permutation_null.parquet',index=False)
display(primary_summary)

## 6. Dimension and investor-representation sensitivity

All five dimensions and six representations are reported. These are secondary diagnostics; the primary specification remains All-S12 RS-Binary 10D.

In [ ]:
plot=metrics.pivot(index='dimension',columns='representation',values='pair_accuracy')
ax=plot.plot(marker='o',figsize=(11,6));ax.axhline(.5,color='black',ls='--');ax.set_ylabel('Leave-one-pair-out accuracy');ax.set_ylim(0,1);ax.legend(bbox_to_anchor=(1.02,1),loc='upper left');plt.tight_layout();plt.savefig(RESULTS/'pair_accuracy_by_dimension.png',dpi=180,bbox_inches='tight');plt.show()
display(plot)

## 7. Penalty sensitivity

The ridge penalty is not selected from the test results. We display a fixed grid to determine whether conclusions depend on numerical regularization.

In [ ]:
penalty_rows=[]
for penalty in [.01,.1,1.,10.,100.]:
    result,_=mc.evaluate_representation(representations['all_s12_rs_binary'],pairs,'all_s12_rs_binary',penalty=penalty,dimensions=(10,));penalty_rows.append(result)
penalty_sensitivity=pd.concat(penalty_rows,ignore_index=True);penalty_sensitivity.to_parquet(RESULTS/'penalty_sensitivity.parquet',index=False)
display(penalty_sensitivity)

## 8. Exact-SIC2-pair robustness

The strongest matching restriction retains only pairs sharing exact SIC2. This improves comparability but reduces sample size and power.

In [ ]:
exact_pairs=pairs[pairs.same_sic2].reset_index(drop=True).copy();exact_pairs['pair_id']=np.arange(1,len(exact_pairs)+1)
exact_metrics=[]
for name,frame in representations.items():
    result,_=mc.evaluate_representation(frame,exact_pairs,name,penalty=1.,dimensions=(10,));exact_metrics.append(result)
exact_metrics=pd.concat(exact_metrics,ignore_index=True);exact_metrics.to_parquet(RESULTS/'exact_sic2_pair_metrics.parquet',index=False)
display(exact_metrics.sort_values('pair_accuracy',ascending=False))

## 9. Industry-residualized embeddings

To remove broad industry location from the representation itself, every coordinate is demeaned within SIC2 using the full pre-event universe. The identical frozen pairs and contrastive protocol are then reused.

In [ ]:
residual_metrics=[]
for name,frame in representations.items():
    residual=dc.residualize_industry(frame,industry,10)
    result,_=mc.evaluate_representation(residual,pairs,name+'_industry_residual',penalty=1.,dimensions=(10,));residual_metrics.append(result)
residual_metrics=pd.concat(residual_metrics,ignore_index=True);residual_metrics.to_parquet(RESULTS/'industry_residualized_pair_metrics.parquet',index=False)
display(residual_metrics.sort_values('pair_accuracy',ascending=False))

## 10. Conventional-characteristic contrastive benchmark

Because matching was based on these characteristics, a characteristic difference model should have little remaining ability to orient held-out pairs. This is a demanding benchmark for incremental embedding structure.

In [ ]:
char_frame=dc.characteristic_frame(ROOT,reference)
char_cols=[c for c in char_frame if c not in ['permco','firm_name']]
char_frame=char_frame.rename(columns={c:f'embedding_{i:03d}' for i,c in enumerate(char_cols,1)})
char_metrics,_=mc.evaluate_representation(char_frame,pairs,'characteristics',penalty=1.,dimensions=(len(char_cols),))
char_metrics.to_parquet(RESULTS/'characteristic_pair_benchmark.parquet',index=False)
display(char_metrics)

## 11. Outcome-free decision gate

Proceed only if the frozen 10D primary model achieves economically meaningful pair accuracy with sign-flip permutation support and remains directionally credible in exact-industry or industry-residualized tests. No return result may alter this decision.

In [ ]:
row=primary_summary.iloc[0]
primary_accuracy=float(row.observed_pair_accuracy);primary_p=float(row.accuracy_permutation_p)
decision='proceed' if primary_accuracy>=.65 and primary_p<=.05 else 'stop_or_reformulate'
gate=pd.DataFrame([{'primary_pair_accuracy':primary_accuracy,'primary_permutation_p':primary_p,'minimum_accuracy':.65,'maximum_p':.05,'decision':decision,'event_returns_loaded':False}])
gate.to_parquet(RESULTS/'outcome_free_decision_gate.parquet',index=False)
display(gate)